In [1]:
#Import pandas
import pandas as pd

#Import LangChain chat model
from langchain_openai import ChatOpenAI

#Import LangChain message type
from langchain_core.messages import HumanMessage

#Import project functions
from backend.video_processor import process_video
from backend.rag_pipeline import ask_video_with_sources


#Create judge model
judge_llm = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=0
)


#Process test video
process_video("https://www.youtube.com/watch?v=3kAiPSEnrHI")


#Evaluation questions
questions = [
    "What is the main topic of the video?",
    "Which habit was mentioned first?",
    "What does the video say about caffeine?",
    "What does the video say about bedroom temperature?",
    "What does the video say about noise and sleep?",
]


#Store evaluation rows
rows = []


#Loop through questions
for question in questions:

    #Get RAG answer
    result = ask_video_with_sources(question)

    #Create judge prompt
    judge_prompt = f"""
You are an evaluator for a RAG-based YouTube video Q&A system.

Evaluate the answer using ONLY the provided sources.

Evaluation criteria:

1. Faithfulness:
Is the answer supported by the provided sources?

2. Relevance:
Does the answer directly answer the question?

3. Clarity:
Is the answer clear and easy to understand?

4. Hallucination Safety:
Does the answer add unsupported claims that are not grounded in the sources?

Question:
{question}

Answer:
{result["answer"]}

Sources:
{result["sources"]}

Return EXACTLY in this format:

Faithfulness: X/5
Relevance: X/5
Clarity: X/5
Hallucination Safety: X/5
Feedback: short feedback
"""

    #Run judge evaluation
    evaluation = judge_llm.invoke([
        HumanMessage(content=judge_prompt)
    ])

    #Save row
    rows.append({
        "Question": question,
        "Answer": result["answer"],
        "Sources": len(result["sources"]),
        "Evaluation": evaluation.content
    })


#Create dataframe
df = pd.DataFrame(rows)


#Display dataframe
display(df)

KeyboardInterrupt: 